<a href="https://colab.research.google.com/github/aditya-singh-231/pytorch-campusx/blob/main/MNIST_dataset_NN_vs_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torchvision import datasets
import matplotlib.pyplot as plt


import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import os
import random

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

In [ ]:
transform = transforms.ToTensor()

mnist_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

print("Total MNIST samples:", len(mnist_dataset))

Total MNIST samples: 60000


In [ ]:
selected_indices = []

targets = np.array(mnist_dataset.targets)

for class_id in range(10):


    class_indices = np.where(targets == class_id)[0]


    selected = np.random.choice(
        class_indices,
        size=50,
        replace=False
    )

    selected_indices.extend(selected)


np.random.shuffle(selected_indices)

print("Number of selected samples:", len(selected_indices))

Number of selected samples: 500


/tmp/ipykernel_1466/1283588616.py:3: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  targets = np.array(mnist_dataset.targets)


In [ ]:
X = []
y = []

for idx in selected_indices:

    image, label = mnist_dataset[idx]

    X.append(image)
    y.append(label)

X = torch.stack(X)
y = torch.tensor(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([500, 1, 28, 28])
y shape: torch.Size([500])


In [ ]:
dataset = {
    "images": X,
    "labels": y
}

dataset_path = "./mnist_500.pt"

torch.save(dataset, dataset_path)

print("Dataset saved at:", os.path.abspath(dataset_path))

Dataset saved at: /content/mnist_500.pt


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 400
Testing samples: 100


In [ ]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
class NeuralNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.network = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),

            #nn.Linear(128, 64),
            #nn.ReLU(),

            nn.Linear(512, 10),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

In [ ]:
model= NeuralNetwork()

print(model)
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=10, bias=True)
    (3): Softmax(dim=1)
  )
)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=10, bias=True)
    (3): Softmax(dim=1)
  )
)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)


        outputs = model(images)


        loss = criterion(outputs, labels)


        optimizer.zero_grad()


        loss.backward()


        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{num_epochs}], "
        f"Loss: {avg_loss:.4f}"
    )

Epoch [1/10], Loss: 2.2338
Epoch [2/10], Loss: 1.9613
Epoch [3/10], Loss: 1.7521
Epoch [4/10], Loss: 1.6377
Epoch [5/10], Loss: 1.5810
Epoch [6/10], Loss: 1.5476
Epoch [7/10], Loss: 1.5395
Epoch [8/10], Loss: 1.5171
Epoch [9/10], Loss: 1.5073
Epoch [10/10], Loss: 1.4975


In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

In [ ]:
nn_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

nn_precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted"
)

nn_recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print("Neural Network Results")
print("----------------------")
print(f"Accuracy : {nn_accuracy:.4f}")
print(f"Precision: {nn_precision:.4f}")
print(f"Recall   : {nn_recall:.4f}")

Neural Network Results
----------------------
Accuracy : 0.8800
Precision: 0.8944
Recall   : 0.8800


In [ ]:
X_train_svm = X_train.numpy().reshape(
    len(X_train),
    -1
)

X_test_svm = X_test.numpy().reshape(
    len(X_test),
    -1
)

y_train_svm = y_train.numpy()
y_test_svm = y_test.numpy()

print("SVM training shape:", X_train_svm.shape)
print("SVM testing shape:", X_test_svm.shape)

SVM training shape: (400, 784)
SVM testing shape: (100, 784)


In [ ]:
svm_model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale"
)

svm_model.fit(
    X_train_svm,
    y_train_svm
)

SVC(C=10)

In [ ]:
svm_predictions = svm_model.predict(
    X_test_svm
)

In [ ]:
svm_accuracy = accuracy_score(
    y_test_svm,
    svm_predictions
)

svm_precision = precision_score(
    y_test_svm,
    svm_predictions,
    average="weighted"
)

svm_recall = recall_score(
    y_test_svm,
    svm_predictions,
    average="weighted"
)

print("SVM Results")
print("-----------")
print(f"Accuracy : {svm_accuracy:.4f}")
print(f"Precision: {svm_precision:.4f}")
print(f"Recall   : {svm_recall:.4f}")

SVM Results
-----------
Accuracy : 0.9200
Precision: 0.9321
Recall   : 0.9200


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Neural Network",
        "SVM"
    ],

    "Accuracy": [
        nn_accuracy,
        svm_accuracy
    ],

    "Precision": [
        nn_precision,
        svm_precision
    ],

    "Recall": [
        nn_recall,
        svm_recall
    ]
})

print(results)

            Model  Accuracy  Precision  Recall
0  Neural Network      0.88   0.894351    0.88
1             SVM      0.92   0.932136    0.92


In [ ]:
print("\nModels Comparison: NN vs SVM")

if nn_accuracy > svm_accuracy:
    print("Neural Network has higher accuracy.")
elif svm_accuracy > nn_accuracy:
    print("SVM has higher accuracy.")
else:
    print("Both models have the same accuracy.")

if nn_precision > svm_precision:
    print("Neural Network has higher precision.")
elif svm_precision > nn_precision:
    print("SVM has higher precision.")
else:
    print("Both models have the same precision.")

if nn_recall > svm_recall:
    print("Neural Network has higher recall.")
elif svm_recall > nn_recall:
    print("SVM has higher recall.")
else:
    print("Both models have the same recall.")


Models Comparison: NN vs SVM
SVM has higher accuracy.
SVM has higher precision.
SVM has higher recall.
